# Recoverable 16,000-Step Single-Formula Symbolic Regression: Iteration 1

## TL;DR

This notebook is a minimally changed replacement for the stalled version-06 shape search. It preserves the original mathematical structure, 27 shape inputs, operators, complexity limits, 50,000-row evolutionary batches and the total `2,000 x 8 = 16,000` population-iteration target.

The final result is still one shape formula:

`stress = case_mean + exp(case_log_scale) * single_shape_formula`

Only execution control is changed. The search is divided into 20 sequential warm-start segments. Each segment runs in an isolated process, writes a checkpoint and is watched for file activity. A stalled worker is stopped and retried from its checkpoint, while all previously completed segments remain saved.

## Context and Scope

### What remains unchanged

- The completed Iteration-1 case-mean and case-log-scale formulas are reused from version 06.
- Every one of the 119 training cases contributes the same locked 5,000-row discovery design, for 595,000 discovery rows.
- Shape evolution uses all 595,000 rows as the available training pool and a 50,000-row PySR batch per fitness cycle.
- The single shape formula uses the same 27 local, within-case-standardised and case-context predictors.
- The operator set remains `+`, `-`, `*`, `/`, `square`, `cube` and `abs`; `maxsize=28` and `maxdepth=10`.
- Formula selection reads all elements in all 15 validation cases. The 15 internal-test cases are evaluated once after selection. The 50 final-test cases remain sealed.

### What changes to prevent a freeze

- One 16,000-progress process is replaced by 20 warm-start segments of 800 population iterations each. Cumulative planned progress is written to `search_progress.json`.
- Julia uses multithreading inside one isolated worker instead of a persistent multiprocessing worker pool.
- The parent controller stops a segment after 45 minutes without file activity or after three hours of wall time, then returns to the last stable segment checkpoint.
- Every successful segment stores a separate stable checkpoint snapshot, so a checkpoint interrupted during writing is not trusted.
- Worker memory is released when each segment exits. Rerunning this notebook skips completed segments.

Only successfully completed segments count toward 16,000. A failed attempt can add wasted wall time, but it is excluded from the accepted cumulative search state and recorded in the attempt audit.

## 1. Fresh Kernel and Setup

Run this notebook in a fresh NotebookCT3 kernel. Stop any older PySR notebook first so its Julia processes do not compete for CPU or memory. On macOS, the controller starts `caffeinate` while the run is active, but the laptop lid should still remain open and power connected.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


def resolve_package_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'src' / 'recoverable_single_shape_symbolic.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate the NotebookCT3 package root.')


PACKAGE_ROOT = resolve_package_root()
if str(PACKAGE_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT / 'src'))

from recoverable_single_shape_symbolic import (
    RecoverableSingleShapeConfig,
    package_output_dir,
    preflight_recoverable_iteration,
    run_recoverable_iteration_one,
)

print('Package root:', PACKAGE_ROOT)
print('Python:', sys.executable)

Package root: /Users/novwin/Documents/University/University of Manchester/毕设项目/Nuclear graphite/NotebookCT3
Python: /usr/local/bin/python3.12


## 2. Locked Search Configuration

`RUN_RECOVERABLE_SEARCH=True` starts or resumes the search. Do not change `output_subdir` between retries. A deliberate change to features or search settings must use a new output subdirectory so incompatible checkpoints cannot be mixed.

Engineering acceptance gates are still calculated and exported as diagnostics, but they are not hard rejection rules in this time-limited prototype. Selection first keeps candidates within 3% of the best validation macro RMSE and then prefers the simpler formula, using validation P99 error as a tie-break.

In [2]:
RUN_RECOVERABLE_SEARCH = True

CONFIG = RecoverableSingleShapeConfig(
    iteration=1,
    output_subdir='iteration_1',
    rows_per_training_case=5_000,
    total_niterations=2_000,
    populations=8,
    segment_niterations=100,
    population_size=40,
    ncycles_per_iteration=100,
    batch_size=50_000,
    maxsize=28,
    maxdepth=10,
    julia_threads=8,
    no_activity_timeout_seconds=45 * 60,
    segment_wall_timeout_seconds=3 * 60 * 60,
    watchdog_poll_seconds=60,
    max_attempts_per_segment=3,
    max_shape_candidates_for_full_validation=14,
    include_old_partial_frontier=True,
    force_rebuild_training_cache=False,
)

OUTPUT_DIR = package_output_dir(PACKAGE_ROOT, CONFIG)
print('Output directory:', OUTPUT_DIR)
print('Segments:', CONFIG.n_segments)
print('Population iterations per segment:', CONFIG.population_iterations_per_segment)
print('Planned cumulative population iterations:', CONFIG.target_population_iterations)
display(pd.DataFrame([CONFIG.__dict__]).T.rename(columns={0: 'value'}))

Output directory: /Users/novwin/Documents/University/University of Manchester/毕设项目/Nuclear graphite/NotebookCT3/outputs/08_recoverable_16000_single_formula/iteration_1
Segments: 20
Population iterations per segment: 800
Planned cumulative population iterations: 16000


,value
iteration,1
output_subdir,iteration_1
rows_per_training_case,5000
total_niterations,2000
populations,8
segment_niterations,100
population_size,40
ncycles_per_iteration,100
batch_size,50000
maxsize,28


## 3. Preflight

This check confirms the frozen 119/15/15 development split, 50 sealed final-test cases, 27 shape features, completed case-level formulas, worker script and exact 16,000 planned search target. It does not read final-test element data.

In [3]:
PREFLIGHT = preflight_recoverable_iteration(PACKAGE_ROOT, CONFIG)
display(PREFLIGHT['checks'])

print('Train cases:', len(PREFLIGHT['inputs']['train_ids']))
print('Validation cases:', len(PREFLIGHT['inputs']['validation_ids']))
print('Internal-test cases:', len(PREFLIGHT['inputs']['internal_ids']))
print('Sealed final-test cases:', len(PREFLIGHT['inputs']['final_ids']))

,check,value,expected,pass
0,training_cases,119,119,True
1,validation_cases,15,15,True
2,internal_test_cases,15,15,True
3,sealed_final_cases,50,50,True
4,shape_features,27,27,True
5,worker_script_exists,True,True,True
6,target_population_iterations,16000,16000,True
7,segments,20,20,True


Train cases: 119
Validation cases: 15
Internal-test cases: 15
Sealed final-test cases: 50


## 4. Run or Resume

Run this cell once and leave it active. It prints a short watchdog message about every five minutes. Detailed PySR logs are stored under `segments/segment_XX/attempt_Y/`.

If the kernel, VS Code or computer is interrupted, rerun the notebook from the top. Completed `complete.json` markers are skipped and the unfinished segment resumes from the shared checkpoint. Do not manually delete the shared checkpoint while keeping segment markers.

In [4]:
RESULT = None
if RUN_RECOVERABLE_SEARCH:
    RESULT = run_recoverable_iteration_one(PACKAGE_ROOT, CONFIG)
    display(pd.DataFrame([RESULT]))
else:
    print('Search skipped because RUN_RECOVERABLE_SEARCH=False.')

Segment 1/20, attempt 1: starting (fresh search)
Segment 1/20, attempt 1: elapsed=0.0 min, inactive=0.0 min, checkpoint=False
Segment 1/20, attempt 1: elapsed=5.0 min, inactive=0.0 min, checkpoint=True
Segment 1/20, attempt 1: elapsed=10.0 min, inactive=0.0 min, checkpoint=True
Segment 1/20, attempt 1: elapsed=15.0 min, inactive=0.0 min, checkpoint=True
Segment 1/20: complete; planned cumulative progress 800/16000
Segment 2/20, attempt 1: starting (checkpoint resume)
Segment 2/20, attempt 1: elapsed=0.0 min, inactive=0.0 min, checkpoint=True
Segment 2/20, attempt 1: elapsed=5.0 min, inactive=0.0 min, checkpoint=True
Segment 2/20, attempt 1: elapsed=10.0 min, inactive=0.0 min, checkpoint=True
Segment 2/20, attempt 1: elapsed=15.0 min, inactive=0.0 min, checkpoint=True
Segment 2/20: complete; planned cumulative progress 1600/16000
Segment 3/20, attempt 1: starting (checkpoint resume)
Segment 3/20, attempt 1: elapsed=0.0 min, inactive=0.0 min, checkpoint=True
Segment 3/20, attempt 1: elap

/Users/novwin/Documents/University/University of Manchester/毕设项目/Nuclear graphite/NotebookCT3/src/recoverable_single_shape_symbolic.py:766: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[False False False  True False False False False False  True False  True
  True  True]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  candidate_metrics.loc[scored.index, scored.columns] = scored
/Users/novwin/Documents/University/University of Manchester/毕设项目/Nuclear graphite/NotebookCT3/src/recoverable_single_shape_symbolic.py:766: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[False False False False False False False False False False False False
 False False]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  candidate_metrics.loc[scored.index, scored.

[2/15] internal_test full-case symbolic evaluation: case_28
[3/15] internal_test full-case symbolic evaluation: case_57
[4/15] internal_test full-case symbolic evaluation: case_66
[5/15] internal_test full-case symbolic evaluation: case_71
[6/15] internal_test full-case symbolic evaluation: case_82
[7/15] internal_test full-case symbolic evaluation: case_93
[8/15] internal_test full-case symbolic evaluation: case_98
[9/15] internal_test full-case symbolic evaluation: case_102
[10/15] internal_test full-case symbolic evaluation: case_117
[11/15] internal_test full-case symbolic evaluation: case_147
[12/15] internal_test full-case symbolic evaluation: case_159
[13/15] internal_test full-case symbolic evaluation: case_196
[14/15] internal_test full-case symbolic evaluation: case_197
[15/15] internal_test full-case symbolic evaluation: case_199


,status,iteration,prototype_only,formula_shape_components,train_cases,validation_cases,internal_test_cases,final_test_cases_read,training_rows,planned_population_iterations,completed_segments,selected_shape_candidate,selection_pool_status,elapsed_seconds,output_directory
0,complete,1,True,1,119,15,15,0,595000,16000,20,5,prototype_engineering_gates_diagnostic_only,26406.850401,/Users/novwin/Documents/University/University ...


## 5. Progress and Saved Results

During the search, `search_progress.json` is the clearest cumulative indicator. Each successfully completed segment adds 800 planned population iterations. `segment_attempt_audit.csv` records stalls, timeouts, retries and checkpoint recovery.

After all 20 segments, the notebook validates shortlisted formulas on complete cases and exports one combined symbolic formula. P95/P99, hotspot and engineering-gate fields remain available for interpretation even though they are not mandatory acceptance filters in this prototype.

In [5]:
progress_path = OUTPUT_DIR / 'search_progress.json'
if progress_path.exists():
    display(pd.DataFrame([json.loads(progress_path.read_text(encoding='utf-8'))]))

attempt_path = OUTPUT_DIR / 'segment_attempt_audit.csv'
if attempt_path.exists():
    display(pd.read_csv(attempt_path).tail(20))

artifacts = {
    'completion': OUTPUT_DIR / 'round_complete.json',
    'formula': OUTPUT_DIR / 'selected_composite_formula.csv',
    'formula_text': OUTPUT_DIR / 'selected_composite_formula.txt',
    'split_metrics': OUTPUT_DIR / 'selected_formula_split_metrics.csv',
    'candidate_metrics': OUTPUT_DIR / 'shape_candidate_validation_metrics.csv',
}
display(pd.DataFrame([
    {'artifact': name, 'exists': path.exists(), 'path': str(path)}
    for name, path in artifacts.items()
]))

if artifacts['completion'].exists():
    print(artifacts['formula_text'].read_text(encoding='utf-8'))
    display(pd.read_csv(artifacts['split_metrics']))
    candidates = pd.read_csv(artifacts['candidate_metrics'])
    display(candidates.sort_values(
        ['selected_candidate', 'validation_macro_rmse'],
        ascending=[False, True],
    ).head(20))
else:
    print('The run is not complete. The same notebook can be rerun safely.')

,completed_segments,contiguous_completed_segments,total_segments,planned_population_iterations_completed,target_population_iterations,planned_progress_fraction,note,updated_at
0,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",20,20,16000,16000,1.0,Only successful segments count toward the plan...,2026-08-18T06:43:26+0100


,segment,attempt,resume_from_checkpoint,return_code,termination_reason,stalled,wall_timed_out,elapsed_seconds,checkpoint_exists_after,hall_of_fame_exists_after,worker_frontier_exists,finished_at
0,1,1,False,0,process_exit,False,False,1020.613604,True,True,True,2026-08-17T23:40:26+0100
1,2,1,True,0,process_exit,False,False,960.517708,True,True,True,2026-08-17T23:56:27+0100
2,3,1,True,0,process_exit,False,False,840.228686,True,True,True,2026-08-18T00:10:27+0100
3,4,1,True,0,process_exit,False,False,840.175735,True,True,True,2026-08-18T00:24:27+0100
4,5,1,True,0,process_exit,False,False,900.261690,True,True,True,2026-08-18T00:39:28+0100
5,6,1,True,0,process_exit,False,False,1020.552497,True,True,True,2026-08-18T00:56:28+0100
6,7,1,True,0,process_exit,False,False,960.441248,True,True,True,2026-08-18T01:12:29+0100
7,8,1,True,0,process_exit,False,False,1020.679911,True,True,True,2026-08-18T01:29:29+0100
8,9,1,True,0,process_exit,False,False,1020.514243,True,True,True,2026-08-18T01:46:30+0100
9,10,1,True,0,process_exit,False,False,1501.554171,True,True,True,2026-08-18T02:11:32+0100


,artifact,exists,path
0,completion,True,/Users/novwin/Documents/University/University ...
1,formula,True,/Users/novwin/Documents/University/University ...
2,formula_text,True,/Users/novwin/Documents/University/University ...
3,split_metrics,True,/Users/novwin/Documents/University/University ...
4,candidate_metrics,True,/Users/novwin/Documents/University/University ...


CT3 hierarchical symbolic stress formula

Iteration: 1

1. Case mean stress
mu = -0.0116301024532845*temperature_mean + 0.0470427355318997*temperature_p95 - 3.16087946558646*weight_loss_rate_mean + 0.103449215368151*z_max - 0.870195660324862*Abs(4.60760803334225*fluence_rate_p95 - 20.3572633494299) - 100.195330145459

2. Positive case stress scale
scale = exp(-1.47091131932291*rho_mean - 1011.416708227*theta_sin_std - 1.1690770556861*weight_loss_rate_mean + 0.904578545888617*z_mean + 9.23869712445276)

3. Normalised element-level spatial shape
shape = 1.10317833861391*(0.55673575*(0.0509611749551904*rho - 9.32241465638866)*(0.0509611749551904*rho - 8.66867535638866) - 31.9731949759449*(theta_cos - 0.922666019258146)**2)*(Abs(0.0509611749551904*rho - 8.59575422638866) - 2.4201684) + 0.285739561816426

4. Combined deployable expression
sigma = (-0.0116301024532845*temperature_mean + 0.0470427355318997*temperature_p95 - 3.16087946558646*weight_loss_rate_mean + 0.103449215368151*z_max - 0.

,iteration,split,model,n_cases,n_elements_evaluated,micro_mae,micro_rmse,micro_r2,macro_mae,macro_rmse,...,mean_top5_actual_rmse,mean_top5_actual_bias,mean_p95_relative_error,mean_p95_underprediction_fraction,mean_p99_relative_error,mean_p99_underprediction_fraction,mean_top5pct_hotspot_overlap,mean_top1pct_hotspot_overlap,mean_top1_recall_in_predicted_top5,max_prediction_abs_max_ratio
0,1,internal_test,hierarchical_symbolic,15,6005400,1.847777,2.471889,0.281895,1.847777,2.451950,...,5.110821,-3.972092,0.150652,0.150652,0.253503,0.253503,0.319945,0.447236,0.60959,1.166592
1,1,validation,hierarchical_symbolic,15,6005400,1.768228,2.403750,0.358755,1.768228,2.347327,...,4.630008,-3.574538,0.128592,0.118436,0.214801,0.214801,0.307007,0.488994,0.60020,0.838627


,stage,run_id,candidate_index,complexity,loss,score,equation,formula_scaled_sympy,formula_original_variables,feature_support_json,...,gate_p95_spearman,gate_p99_spearman,gate_p95_spread,gate_p99_spread,gate_numerical_guardrail,all_provisional_acceptance_gates_pass,selected_candidate,engineering_gates_used_as_hard_filter,selection_pool_status,selection_method
11,shape,recoverable_i1_single_shape,5,22,0.545558,0.028147,((square(theta_cos_scaled) * -0.1345017) - ((r...,(-(-0.55673575)*rho_scaled*(rho_scaled + 0.653...,1.10317833861391*(0.55673575*(0.05096117495519...,"[""rho_scaled"", ""theta_cos_scaled""]",...,True,True,True,True,True,False,True,False,prototype_engineering_gates_diagnostic_only,finite_full_validation_candidates; within_3pct...
3,shape,hierarchical_i1_shape_20260817_061209,29,7,0.793274,NaN,((fluence_rate_within_case_z_scaled * fluence_...,-0.24032879*fluence_rate_scaled*fluence_rate_w...,-4.08771545322238*theta_cos - 0.26512551527329...,"[""fluence_rate_scaled"", ""fluence_rate_within_c...",...,True,True,True,False,True,False,False,False,prototype_engineering_gates_diagnostic_only,finite_full_validation_candidates; within_3pct...
12,shape,recoverable_i1_single_shape,2,25,0.510062,0.001265,(((abs(theta_cos_scaled) * -0.36819604) - (((r...,(-(-0.59498614)*rho_scaled*(rho_scaled + 0.529...,1.10317833861391*(0.59498614*(0.05096117495519...,"[""rho_scaled"", ""theta_cos_scaled""]",...,True,True,True,True,True,False,False,False,prototype_engineering_gates_diagnostic_only,finite_full_validation_candidates; within_3pct...
13,shape,recoverable_i1_single_shape,0,27,0.490487,0.026093,(((abs(theta_cos_scaled) * -0.3372796) - ((rho...,-(-0.14733057)*z_within_case_z_scaled + (-(-0....,0.157474565807974*z_within_case_z + 1.10317833...,"[""rho_scaled"", ""theta_cos_scaled"", ""z_within_c...",...,True,True,True,True,True,False,False,False,prototype_engineering_gates_diagnostic_only,finite_full_validation_candidates; within_3pct...
9,shape,hierarchical_i1_shape_20260817_061209,11,19,0.578649,NaN,(((abs(rho_scaled) - 1.6013404) * (abs(theta_c...,-0.54873085*(-rho_scaled + Abs(theta_cos_scale...,-0.605347987449196*(Abs(0.0509611749551904*rho...,"[""rho_scaled"", ""theta_cos_scaled""]",...,True,True,True,True,True,False,False,False,prototype_engineering_gates_diagnostic_only,finite_full_validation_candidates; within_3pct...
8,shape,hierarchical_i1_shape_20260817_061209,14,17,0.591968,NaN,(((abs(rho_scaled) - 1.5967752) * (abs(theta_c...,-0.54832864*(-rho_scaled + Abs(theta_cos_scale...,-0.604904278089622*(Abs(0.0509611749551904*rho...,"[""rho_scaled"", ""theta_cos_scaled""]",...,True,True,True,True,True,False,False,False,prototype_engineering_gates_diagnostic_only,finite_full_validation_candidates; within_3pct...
10,shape,recoverable_i1_single_shape,8,21,0.561132,0.014049,(abs(rho_scaled + 0.73987824) + -2.419068) * (...,(-(-0.43886876)*rho_scaled*rho_scaled + Abs(th...,1.10317833861391*(38.1409494671124*(0.00546652...,"[""rho_scaled"", ""theta_cos_scaled""]",...,True,True,True,True,True,False,False,False,prototype_engineering_gates_diagnostic_only,finite_full_validation_candidates; within_3pct...
7,shape,recoverable_i1_single_shape,15,16,0.596511,0.023588,((theta_cos_scaled * (theta_sin_scaled * -0.54...,(-rho_scaled + theta_cos_scaled*theta_sin_scal...,-0.629809808770704*(Abs(0.0509611749551904*rho...,"[""rho_scaled"", ""theta_cos_scaled"", ""theta_sin_...",...,True,True,True,True,True,False,False,False,prototype_engineering_gates_diagnostic_only,finite_full_validation_candidates; within_3pct...
6,shape,hierarchical_i1_shape_20260817_061209,19,13,0.680355,NaN,(rho_scaled - square(theta_cos_scaled)) * ((sq...,(rho_scaled - theta_cos_scaled**2)*(0.10390691...,1.10317833861391*(9.03028138593995*(0.00546652...,"[""rho_scaled"", ""theta_cos_scaled""]",...,True,True,True,True,True,False,False,False,prototype_engineering_gates_diagnostic_only,finite_full_validation_candidates; within_3pct...
5,shape,hierarchical_i1_shape_20260817_061209,

## Takeaways and Interpretation Boundary

This notebook tests whether the original single-shape symbolic design can complete reliably under the available time and hardware. The selected equation is an Iteration-1 prototype, not yet an engineering-qualified lifetime model. It predicts FEM maximum-principal stress when post-deformation coordinates and the three parameter fields are available.

Lifetime conversion still requires a professor-confirmed strength, damage or failure relationship. The sealed 50-case final test must not be used until formula structure and constants have been fixed across the development workflow.